In [2]:
from pathlib import Path

import pandas as pd
import torch
from pytorch_forecasting import TimeSeriesDataSet

# Import the loader (and its safe-globals setup) from the loader module.
from load_tft_model import load_tft_checkpoint_safely


def load_time_series_dataset(path, trusted_source=True, map_location="cpu"):
    """
    Loads a TimeSeriesDataSet pickle directly, bypassing
    TimeSeriesDataSet.load()'s internal torch.load() call.

    TimeSeriesDataSet.load() calls torch.load(fname) with no weights_only
    argument, so on PyTorch 2.6+ it silently defaults to weights_only=True.
    That fails as soon as it hits any nested custom class (GroupNormalizer,
    NaNLabelEncoder, TorchNormalizer, sklearn encoders, etc.) that isn't on
    the safe-globals list -- and allowlisting every nested class individually
    is impractical. Since this is a local file you generated yourself,
    loading with weights_only=False (full unpickling) is the pragmatic fix.
    Only do this for files whose origin you trust.
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Dataset file not found at: {path.resolve()}")

    if trusted_source:
        obj = torch.load(str(path), map_location=map_location, weights_only=False)
    else:
        # Will very likely fail on nested classes; kept for completeness.
        obj = torch.load(str(path), map_location=map_location, weights_only=True)

    if not isinstance(obj, TimeSeriesDataSet):
        raise TypeError(
            f"Loaded object from {path} is not a TimeSeriesDataSet "
            f"(got {type(obj)!r}). The .pkl may be corrupted or from an "
            "incompatible pytorch_forecasting version."
        )
    return obj

# =====================================================================
# 1. Paths
# =====================================================================
ckpt_path = Path("models/tft/tft_model.ckpt")
dataset_pkl_path = Path("models/tft/training_dataset.pkl")
test_parquet_path = Path("data/features_5Hz/year2025_round01.parquet")

# =====================================================================
# 2. Load model and saved dataset structure
# =====================================================================
model = load_tft_checkpoint_safely(ckpt_path, map_location="cpu", trusted_source=True)
model.eval()

training_dataset = load_time_series_dataset(dataset_pkl_path, trusted_source=True)

# =====================================================================
# 3. Load and format test DataFrame
# =====================================================================
test_df = pd.read_parquet(test_parquet_path)

FLOAT_COLS = [
    "remaining_distance", "distance_into_lap", "speed", "throttle", "brake_pct",
    "gear", "drs", "air_temp", "track_temp", "humidity", "wind_speed", "tyre_life",
    "tyre_age_at_stint_start", "time_since_status_change", "remaining_lap_time",
]
CATEGORY_COLS = [
    "session_id", "driver", "team", "circuit", "compound_at_stint_start",
    "event_format", "track_status",
]

for col in FLOAT_COLS:
    if col in test_df.columns:
        test_df[col] = test_df[col].astype("float32")
for col in CATEGORY_COLS:
    if col in test_df.columns:
        test_df[col] = test_df[col].astype("category")

# --- Sanity check: warn about categories not seen during training.
# TimeSeriesDataSet.from_dataset reuses the training encoders, so unseen
# categories will either raise or get mapped to an "unknown" bucket
# depending on add_nan/allow_missing_timesteps settings -- better to know
# up front than to debug a silent encoder issue later.
for col in CATEGORY_COLS:
    if col in test_df.columns and col in training_dataset.categorical_encoders:
        known = set(training_dataset.categorical_encoders[col].classes_)
        seen = set(test_df[col].astype(str).unique())
        unseen = seen - known
        if unseen:
            print(f"[warning] Column '{col}' has unseen categories not in training data: {sorted(unseen)}")

# =====================================================================
# 4. Create test dataset / loader
# =====================================================================
test_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset,
    test_df,
    predict=True,
    stop_randomization=True,
)

test_loader = test_dataset.to_dataloader(
    train=False,
    batch_size=512,
    num_workers=0,
)

# =====================================================================
# 5. Predict quantiles (P10, P50, P90)
# =====================================================================
predictions = model.predict(
    test_loader,
    mode="quantiles",
    return_x=True,
    return_y=True,
)

# Guard against a silently-mismatched output head (see loader warnings):
# expected shape is (n_samples, n_timesteps, n_quantiles).
n_quantiles = predictions.output.shape[-1]
if n_quantiles < 3:
    raise RuntimeError(
        f"Expected at least 3 quantile outputs (P10/P50/P90) but got {n_quantiles}. "
        "This likely means the model's output_size/loss hparams didn't load "
        "correctly -- check the [load_tft_checkpoint_safely] warnings above."
    )

p10 = predictions.output[:, 0, 0]
p50 = predictions.output[:, 0, 1]
p90 = predictions.output[:, 0, 2]
actual = predictions.y[0][:, 0]

mae = (p50 - actual).abs().mean().item()
print(f"Test MAE (P50 vs Actual): {mae:.3f} seconds")

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "d:\Material\Programming\Machine Learning\F1 LapTime Prediction\f1-laptime-prediction\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [3]:
%pip show torch

Name: torch
Version: 2.14.0+cu126
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: 
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License: 
Location: d:\Material\Programming\Machine Learning\F1 LapTime Prediction\f1-laptime-prediction\Lib\site-packages
Requires: filelock, fsspec, jinja2, networkx, setuptools, sympy, typing-extensions
Required-by: lightning, pytorch-forecasting, pytorch-lightning, torchmetrics, torchvision
Note: you may need to restart the kernel to use updated packages.
